<a href="https://colab.research.google.com/github/shalomali/ed-arrival-forecasting/blob/main/notebooks/01_data_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd


url = "https://github.com/shalomali/ed-arrival-forecasting/raw/main/data/raw/dryad_csv.xlsx"

df = pd.read_excel(url, header = None)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,708,709,710,711,712,713,714,715,716,717
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sum of Count,Column Labels,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,<1/1/14,<1/1/14 Total,1-Jan,1-Jan Total,2-Jan,2-Jan Total,3-Jan,3-Jan Total,4-Jan,...,26-Dec Total,27-Dec,27-Dec Total,28-Dec,28-Dec Total,29-Dec,29-Dec Total,30-Dec,30-Dec Total,Grand Total
3,Row Labels,<1/1/14,NaN,Jan,NaN,Jan,NaN,Jan,NaN,Jan,...,NaN,Dec,NaN,Dec,NaN,Dec,NaN,Dec,NaN,NaN
4,<1/1/14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import pandas as pd
import numpy as np

url = "https://github.com/shalomali/ed-arrival-forecasting/raw/main/data/raw/dryad_csv.xlsx"
df = pd.read_excel(url, header=None)

# 2. Dynamically find the row that contains '1-Jan' so we don't rely on a hardcoded row index
date_row_idx = df[df.eq('1-Jan').any(axis=1)].index[0]
date_row = df.iloc[date_row_idx]

# 3. Create a map of the columns we actually want to keep
cols_to_keep = [0]
date_cols = {}

for i, val in enumerate(date_row):
    val_str = str(val).strip()
    # Strictly keep columns that look like '1-Jan' (contain a hyphen)
    # and exclude 'Total' and the '<1/1/14' aggregations
    if '-' in val_str and 'Total' not in val_str and '<' not in val_str:
        if i != 0:
            cols_to_keep.append(i)
            date_cols[i] = val_str

# 4. Extract and forward-fill the Year
df['Year'] = df[0].apply(lambda x: x if str(x) in ['2014', '2015'] else np.nan)
df['Year'] = df['Year'].ffill()

# 5. Filter out the noise to only keep the hourly data rows
valid_hours = [f"{h} {ampm}" for ampm in ['AM', 'PM'] for h in [12] + list(range(1, 12))]
df_data = df[df[0].isin(valid_hours)].copy()

# 6. Apply our column mask to drop the duplicate/summary columns
cols_to_keep.append('Year')
df_clean = df_data[cols_to_keep]

# Rename the columns so they make sense
df_clean = df_clean.rename(columns={0: 'Hour'})
df_clean = df_clean.rename(columns=date_cols)

# 7. Unpivot (Melt) the dataframe from Wide to Long format
df_melted = pd.melt(
    df_clean,
    id_vars=['Year', 'Hour'],
    var_name='DayMonth',
    value_name='Arrivals'
)

# 8. Combine the Year and DayMonth into a true Python Datetime object
df_melted['Date'] = pd.to_datetime(df_melted['DayMonth'] + '-' + df_melted['Year'].astype(str))

# 9. Clean up the final dataframe and sort chronologically
# We will also convert 'Arrivals' to numeric just in case there are hidden strings
df_melted['Arrivals'] = pd.to_numeric(df_melted['Arrivals'], errors='coerce')
df_final = df_melted[['Date', 'Hour', 'Arrivals']].dropna().sort_values(by=['Date', 'Hour']).reset_index(drop=True)

# Preview your clean, modeling-ready data!
df_final.head()

,Date,Hour,Arrivals
0,2014-01-01,1 AM,2.0
1,2014-01-01,1 PM,3.0
2,2014-01-01,10 AM,5.0
3,2014-01-01,10 PM,3.0
4,2014-01-01,11 AM,4.0


In [4]:
# 10. Combine Date and Hour into a single continuous Datetime column
df_final['Datetime'] = pd.to_datetime(df_final['Date'].astype(str) + ' ' + df_final['Hour'])

# 11. Sort by the new true Datetime, and drop the redundant columns
df_final = df_final.sort_values(by='Datetime').reset_index(drop=True)

# Reorder columns to make it clean
df_final = df_final[['Datetime', 'Date', 'Hour', 'Arrivals']]

# Preview the properly sorted data
df_final.head(10)

,Datetime,Date,Hour,Arrivals
0,2014-01-01 00:00:00,2014-01-01,12 AM,2.0
1,2014-01-01 01:00:00,2014-01-01,1 AM,2.0
2,2014-01-01 02:00:00,2014-01-01,2 AM,3.0
3,2014-01-01 03:00:00,2014-01-01,3 AM,1.0
4,2014-01-01 04:00:00,2014-01-01,4 AM,1.0
5,2014-01-01 05:00:00,2014-01-01,5 AM,2.0
6,2014-01-01 06:00:00,2014-01-01,6 AM,2.0
7,2014-01-01 07:00:00,2014-01-01,7 AM,5.0
8,2014-01-01 08:00:00,2014-01-01,8 AM,5.0
9,2014-01-01 09:00:00,2014-01-01,9 AM,6.0
